# 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import utils_z
import geopandas as gpd
import shp_parser as shpar

In [3]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [5]:
city_name = "winnipeg"
city_prefix  = "CA_WP"

# 1. 处理 shapefile 数据并导入数据库

## 1.1 查看数据

In [7]:
test_shp_path = rf"E:\2_data\building_3d_opensource\{city_name}\shp\Buildings_UDT_241125.shp"
gdf = gpd.read_file(test_shp_path)

# 基本信息
print(f"CRS: {gdf.crs}")
print(f"总行数: {len(gdf)}")
print(f"字段名: {list(gdf.columns)}")
print(f"geometry类型: {gdf.geometry.type.unique()}")

# 前几行
print(gdf.head())

CRS: EPSG:32614
总行数: 284444
字段名: ['TARGET_FID', 'url', 'year_built', 'age_AT', 'rooms_AT', 'year_AT', 'livin_AT', 'land_AT', 'value_AT', 'id', 'type_AT', 'foot_area', 'zoning', 'roll_num', 'layer', 'BLDG_POP_2', 'ht_als', 'htSD_als', 'influence', 'address', 'Parcel_ID', 'geometry']
geometry类型: ['Polygon' 'MultiPolygon']
   TARGET_FID                                                url  year_built  \
0         0.0  http://www.winnipegassessment.com/AsmtPub/engl...      1991.0   
1         1.0  http://www.winnipegassessment.com/AsmtPub/engl...      1992.0   
2         2.0  http://www.winnipegassessment.com/AsmtPub/engl...      2008.0   
3         3.0  http://www.winnipegassessment.com/AsmtPub/engl...      2008.0   
4         4.0  http://www.winnipegassessment.com/AsmtPub/engl...      2002.0   

   age_AT  rooms_AT  year_AT  livin_AT  land_AT  value_AT     id  ...  \
0    33.0       6.0   1991.0    1122.0   6870.0  385000.0  850.0  ...   
1    32.0       8.0   1992.0    1588.0   7235.0  48

In [8]:
# 检查有没有高度相关字段
print(gdf.describe())

          TARGET_FID     year_built         age_AT       rooms_AT  \
count  284444.000000  284444.000000  283606.000000  283600.000000   
mean   142340.366508    1814.287948      58.313467       5.545120   
std     82165.505185     516.766876      33.316784       4.236771   
min         0.000000       0.000000       0.000000       0.000000   
25%     71178.750000    1930.000000      37.000000       5.000000   
50%    142372.500000    1958.000000      60.000000       6.000000   
75%    213501.250000    1978.000000      77.000000       6.000000   
max    284619.000000    2023.000000    2021.000000    1960.000000   

             year_AT       livin_AT       land_AT      value_AT  \
count  283606.000000  283606.000000  2.826190e+05  2.833680e+05   
mean     1811.740626    1188.115421  6.665550e+03  1.157601e+06   
std       520.616939     600.177637  2.596855e+04  1.215683e+07   
min         0.000000       0.000000  0.000000e+00  0.000000e+00   
25%      1930.000000     905.000000  3.6520

In [18]:
gdf_4326 = gdf.to_crs(4326)
print(gdf_4326.geometry.iloc[0])
print(gdf_4326.geometry.bounds.describe())

POLYGON ((16.356862091395183 48.2085455476515, 16.356898703615737 48.20854083614809, 16.356887274552655 48.208501173744764, 16.356778930980497 48.208513967757966, 16.356798733845988 48.20844480674989, 16.356803919672625 48.20842671820741, 16.356810637038354 48.20840324178215, 16.356857259787944 48.20839683732379, 16.3568570576198 48.208396126817526, 16.35686659747618 48.20839486566694, 16.356855977805385 48.20835949336232, 16.35684681469934 48.208360691475086, 16.3568434183386 48.20834888268842, 16.356770678576453 48.208358305549886, 16.356778433567545 48.20833810283025, 16.356838215955587 48.20833074230225, 16.35681999422694 48.20826733639671, 16.35681810835116 48.20826292958784, 16.3568002127845 48.208265361695794, 16.356798326417444 48.20825987557387, 16.356741006675602 48.20826744244812, 16.356742826071688 48.20827362114374, 16.35672472846297 48.20827563055023, 16.356727558350975 48.20828462424814, 16.3567266836068 48.208284417557856, 16.356725795414686 48.20828422885882, 16.356724

In [9]:
print(gdf[['ht_als', 'htSD_als']].describe())
print(f"ht_als缺失：{gdf['ht_als'].isna().sum()}")

              ht_als       htSD_als
count  284444.000000  284444.000000
mean        4.691747       0.655295
std         2.214829       0.573468
min         0.000000       0.000000
25%         3.340000       0.350000
50%         4.190000       0.500000
75%         5.720000       0.880000
max       122.840000      49.900000
ht_als缺失：0


In [9]:
print(gdf['geometry'].geom_type.value_counts())

Polygon    1260
Name: count, dtype: int64


In [20]:
gdf_override = gdf.copy()
gdf_override = gdf_override.set_crs(31256, allow_override=True)
gdf_4326 = gdf_override.to_crs(4326)
print(gdf_4326.geometry.bounds.describe())

              minx         miny         maxx         maxy
count  1260.000000  1260.000000  1260.000000  1260.000000
mean     16.355773    48.209493    16.355894    48.209576
std       0.002010     0.001262     0.001980     0.001257
min      16.352316    48.207400    16.352319    48.207401
25%      16.354091    48.208467    16.354259    48.208565
50%      16.355603    48.209422    16.355685    48.209503
75%      16.357994    48.210674    16.358034    48.210702
max      16.359019    48.211895    16.359043    48.211898


## 1.2 变量

In [10]:
block_table_name = f"block.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
#source_srid  = 3857

## 1.3 建表

In [11]:
# LOD1建筑主表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod1_table_name_full} (
        building_id     VARCHAR PRIMARY KEY,
        block_id        VARCHAR,
        citygml_id      VARCHAR,
        geom_2d         GEOMETRY(Polygon, {target_srid}),
        height          FLOAT,
        floor_count     INTEGER,
        function        VARCHAR,
        ground_z        FLOAT
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_table_name}_geom_idx
    ON {lod1_table_name_full} USING GIST (geom_2d);
""", conn=conn)

# LOD1 surface子表（无citygml_id，面由计算生成）
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod1_surface_table_name_full} (
        surface_id      VARCHAR PRIMARY KEY,
        building_id     VARCHAR REFERENCES {lod1_table_name_full}(building_id),
        surface_type    VARCHAR,
        geom_3d         GEOMETRY(PolygonZ, {target_srid})
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_surface_table_name}_building_idx
    ON {lod1_surface_table_name_full} (building_id);
""", conn=conn)

print(city_prefix + " LOD1表创建完成")

CA_WP LOD1表创建完成


In [12]:
# 清空building和surface表，用于需要重新处理的情况
utils_z.run_sql(f"TRUNCATE TABLE {lod1_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_table_name_full}表已清空")
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_surface_table_name_full}表已清空")

lod1.winnipeg_buildings_lod1表已清空
lod1.winnipeg_building_surfaces_lod1表已清空


In [11]:
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)

## 1.4 批量入库

In [13]:
shp_dir = rf"E:\2_data\building_3d_opensource\{city_name}\shp"

In [14]:
import traceback
import glob
conn.rollback()

In [15]:
# 查找目录下所有shp文件
shp_files = glob.glob(rf"{shp_dir}\*.shp")
print(f"找到 {len(shp_files)} 个shp文件")

# 获取当前最大ID，设置计数器初始值
cur = conn.cursor()
cur.execute(f"SELECT MAX(building_id) FROM {lod1_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1
cur.close()
print(f"building_counter起始值：{building_counter}")

total_count = 0
total_files = len(shp_files)

# 动态计算打印间隔（保证大约每10%的进度打印一次）
print_interval = max(1, total_files // 10)

# 依次处理每个shp文件
for idx, shp_path in enumerate(shp_files, 1):
    try:
        gdf = gpd.read_file(shp_path)
        
        # 入库
        count, building_counter = shpar.insert_buildings_shp(
            gdf, conn,
            lod1_table=lod1_table_name_full,
            city_prefix=city_prefix,
            building_counter=building_counter,
            col_height="ht_als",
            col_ground_z=None,  # 无地面高程字段
            col_citygml_id=None
        )
        # count, building_counter = shpar.insert_buildings_shp(
        #     gdf, conn,
        #     lod1_table=lod1_table_name_full,
        #     city_prefix=city_prefix,
        #     building_counter=building_counter,
        #     col_height_top="O_KOTE",
        #     col_height_bottom="HOEHE_DGM",
        #     col_ground_z="HOEHE_DGM",
        #     col_citygml_id="FMZK_ID",
        #     col_floor_count="H_KLASSE",
        #     set_crs=31256
        # )
        
        total_count += count
        
        # 按间隔打印进度（以及最后一个文件）
        if idx % print_interval == 0 or idx == total_files:
            print(f"进度：{idx}/{total_files} 文件，已入库 {total_count} 栋建筑")
            
    except Exception as e:
        print(f"错误处理文件 {shp_path}：{e}")
        traceback.print_exc()

print(f"\n全部完成！总共处理 {total_files} 个文件，入库 {total_count} 栋建筑")

找到 1 个shp文件
building_counter起始值：1
进度：1/1 文件，已入库 282398 栋建筑

全部完成！总共处理 1 个文件，入库 282398 栋建筑


# 2. 叠合、建面

## 2.1 创建surface入库

In [10]:
# 先查当前surface表最大编号
cur = conn.cursor()
cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
base = int(max_sid.split("_S_")[1]) if max_sid else 0
cur.close()

# GroundSurface
utils_z.run_sql(f"""
    INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
    SELECT 
        '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
        building_id,
        'GroundSurface',
        ST_Force3D(ST_Translate(geom_2d, 0, 0, ground_z))
    FROM {lod1_table_name_full};
""", conn=conn)

# 更新base
cur = conn.cursor()
cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
base = int(max_sid.split("_S_")[1]) if max_sid else 0
cur.close()

# RoofSurface
utils_z.run_sql(f"""
    INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
    SELECT 
        '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
        building_id,
        'RoofSurface',
        ST_Force3D(ST_Translate(geom_2d, 0, 0, ground_z + height))
    FROM {lod1_table_name_full};
""", conn=conn)

# 更新base
cur = conn.cursor()
cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
base = int(max_sid.split("_S_")[1]) if max_sid else 0
cur.close()

# WallSurface
utils_z.run_sql(f"""
    INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
    SELECT
        '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
        building_id,
        'WallSurface',
        ST_MakePolygon(ST_MakeLine(ARRAY[
            ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z),
            ST_MakePoint(ST_X(p2), ST_Y(p2), ground_z),
            ST_MakePoint(ST_X(p2), ST_Y(p2), ground_z + height),
            ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z + height),
            ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z)
        ]))
    FROM (
        SELECT
            b.building_id,
            b.ground_z,
            b.height,
            ST_PointN(ST_ExteriorRing(b.geom_2d), n) AS p1,
            ST_PointN(ST_ExteriorRing(b.geom_2d), n + 1) AS p2
        FROM {lod1_table_name_full} b,
        GENERATE_SERIES(1, ST_NPoints(ST_ExteriorRing(b.geom_2d)) - 1) AS n
    ) edges;
""", conn=conn)

# 统计
cur = conn.cursor()
cur.execute(f"""
    SELECT surface_type, COUNT(*) 
    FROM {lod1_surface_table_name_full} 
    GROUP BY surface_type ORDER BY surface_type
""")
rows = cur.fetchall()
for row in rows:
    print(f"  {row[0]}: {row[1]}")

# 计算总数
total = sum(row[1] for row in rows)
print(f"  Total: {total}")
cur.close()

  GroundSurface: 695635
  RoofSurface: 695635
  WallSurface: 6404942
  Total: 7796212


In [16]:
shpar.generate_surfaces_from_buildings(
    conn,
    lod1_table=lod1_table_name_full,
    surface_table=lod1_surface_table_name_full,
    city_prefix=city_prefix
)

# 统计
cur = conn.cursor()
cur.execute(f"""
    SELECT surface_type, COUNT(*) 
    FROM {lod1_surface_table_name_full} 
    GROUP BY surface_type ORDER BY surface_type
""")
rows = cur.fetchall()
for row in rows:
    print(f"  {row[0]}: {row[1]}")

# 计算总数
total = sum(row[1] for row in rows)
print(f"  Total: {total}")
cur.close()

共282398栋建筑待生成surface
已插入surface：2140883  GroundSurface: 282398
  RoofSurface: 282398
  WallSurface: 1578033
  GroundSurface: 282398
  RoofSurface: 282398
  WallSurface: 1578033
  Total: 2142829


## 2.2 与block叠合

In [17]:
# 确认空间索引
print(block_table_name, lod1_table_name_full)
utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_table_name}_geom_idx
    ON {lod1_table_name_full} USING GIST (geom_2d);
""", conn=conn)
print("准备开始空间叠合...")

block.winnipeg_blocks lod1.winnipeg_buildings_lod1
准备开始空间叠合...


In [18]:
utils_z.run_sql(f"""
    UPDATE {lod1_table_name_full} b
    SET block_id = (
        SELECT bl.block_id
        FROM {block_table_name} bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)
print("空间叠合完成")

空间叠合完成


In [19]:
# 检查匹配情况
result = utils_z.run_sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM {lod1_table_name_full};
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

# 检查包含LOD1建筑的街区数量
sql_counts = f"SELECT COUNT(DISTINCT block_id) FROM {lod1_table_name_full} WHERE block_id IS NOT NULL;"
(lod1_count,) = utils_z.run_sql(sql_counts, conn=conn, fetch=True)[0]

print(f"包含 LOD1 建筑的街区数量: {lod1_count}")

总建筑数：282398
成功匹配block：279344
未匹配block：3054
包含 LOD1 建筑的街区数量: 5414
